### Earliest all connected

In [3]:
from typing import List, Optional

class UnionFind:
    def __init__(self, nodes):
        self.parent = {n: n for n in nodes}
        self.rank = {n: 0 for n in nodes}
        self.components = len(nodes)

    def find(self, x):
        # path compression
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False  # already connected
        # union by rank
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1
        self.components -= 1
        return True


def earliest_all_connected(logs: List[str], riders: List[str]) -> Optional[int]:
    """
    Returns the earliest timestamp at which every rider in `riders`
    is connected through the shared-ride network. Returns None if
    they never all become connected.

    Each log entry format:  "<timestamp> <A> shared-ride-with <B>"
    Logs are assumed sorted chronologically.
    """
    if not riders:
        return None
    if len(riders) == 1:
        # A single rider is trivially "all connected" at time 0 (or undefined).
        return 0

    uf = UnionFind(riders)
    rider_set = set(riders)

    for entry in logs:
        parts = entry.split()
        # parts = [timestamp, A, "shared-ride-with", B]
        if len(parts) < 4:
            continue
        ts, a, b = int(parts[0]), parts[1], parts[-1]

        # Ignore riders not in the known set (defensive)
        if a not in rider_set or b not in rider_set:
            continue

        uf.union(a, b)
        if uf.components == 1:
            return ts

    return None  # never fully connected

# Example: 12-digit (millisecond) timestamps
logs = [
    "167000000001 Alice shared-ride-with Bob",
    "167000000003 Charlie shared-ride-with Dan",
    "167000000008 Bob shared-ride-with Charlie",
    "167000000010 Alice shared-ride-with Eve",
    "167000000020 Bob shared-ride-with Dan",
]
riders = ["Alice", "Bob", "Charlie", "Dan", "Eve"]

print(earliest_all_connected(logs, riders))  # -> 167000000010

167000000010
